In [53]:
!pip install MEDS-Inspect

  Using cached polars-1.42.1-py3-none-any.whl.metadata (10 kB)
  Using cached hydra_core-1.3.4-py3-none-any.whl.metadata (5.7 kB)
  Using cached polars_runtime_32-1.42.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached omegaconf-2.3.1-py3-none-any.whl.metadata (4.5 kB)
  Using cached antlr4-python3-runtime-4.9.3.tar.gz (117 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 8.8 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 143.1 MB/s  0:00:00
Using cached polars-1.42.1-py3-none-any.whl (837 kB)
Using cached polars_runtime_32-1.42.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (57.3 MB)
Using cached hydra_core-1.3.4-py3-none-any.whl (155 kB)
Using cached omegaconf-2.3.1-py3-none-any.whl (79 kB)
  Created wheel for antlr4-python3-runtime: filename=antlr4_pyt

In [ ]:
MEDS_Inspect_cache "/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort"

In [ ]:
!MEDS_Inspect port=8052 +initial_path="/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort"

# Imports

In [1]:
import os
import pandas as pd
import subprocess
import numpy as np
import hail as hl
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.api as sm
import statsmodels.formula.api as smf
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
from datetime import datetime

Loading BokehJS ...

In [2]:
import subprocess

cmd = """
source /home/jupyter/load-env.sh >/dev/null
env
"""

result = subprocess.run(
    ["bash", "-lc", cmd],
    capture_output=True,
    text=True,
    check=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

print("WORKSPACE_CDR:", os.environ.get("WORKSPACE_CDR"))
print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET"))
os.environ["WORKSPACE_CDR"] = "wb-silky-artichoke-2408.C2025Q4R6"

WORKSPACE_CDR: None
WORKSPACE_BUCKET: None


In [3]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 300g "
    "--conf spark.driver.maxResultSize=16g "
    "--conf spark.default.parallelism=64 "
    "--conf spark.sql.shuffle.partitions=256 "
    "pyspark-shell"
)

import hail as hl

hl.init(
    master="local[64]",
    idempotent=True,
    default_reference = "GRCh38"
)

/opt/conda/envs/jupyter/lib/python3.12/site-packages/hail/context.py:350: UserWarning:

Using hl.init with a default_reference argument is deprecated. To set a default reference genome after initializing hail, call `hl.default_reference` with an argument to set the default reference genome.



2026-07-12 20:07:35.716 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/conda/envs/jupyter/lib/python3.12/site-packages/hail/backend/py4j_backend.py:75: UserWarning:

Hail was built and tested with Java 11. You are using Java 17.0.19 which is not supported. This may lead to errors. Consider installing Java 11 and setting the JAVA_HOME environment variable.

2026-07-12 20:07:40.652 SparkBackend$: WARN: This Hail JAR was compiled for Spark 3.5.3, running with Spark 3.5.8.
  Compatibility is not guaranteed.
2026-07-12 20:07:40.817 SparkConf: WARN: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
Running on Apache Spark version 3.5.8
SparkUI available at http://eae9db0b0a0f:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.137-733ac4ccd943
LOGGING: writing

In [ ]:
hl.stop()
hl.init(default_reference = "GRCh38")

# Clinical Data

In [8]:
dataset_08947253_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id"""

dataset_08947253_person_df = pd.read_gbq(
    dataset_08947253_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_08947253_person_df.head(5)

/tmp/ipykernel_1035/3741539726.py:28: FutureWarning:

read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq



Downloading:   0%|          |

,person_id,gender_concept_id,gender,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth
0,5161033,1177221,I prefer not to answer,1967-06-15 00:00:00+00:00,903096,PMI: Skip,903096,PMI: Skip,1177221,I prefer not to answer
1,9498616,1177221,I prefer not to answer,1966-06-15 00:00:00+00:00,903096,PMI: Skip,903096,PMI: Skip,1177221,I prefer not to answer
2,3249566,1177221,I prefer not to answer,1947-06-15 00:00:00+00:00,903096,PMI: Skip,903096,PMI: Skip,1177221,I prefer not to answer
3,1686051,1177221,I prefer not to answer,1994-06-15 00:00:00+00:00,903096,PMI: Skip,903096,PMI: Skip,1177221,I prefer not to answer
4,9912486,1177221,I prefer not to answer,1958-06-15 00:00:00+00:00,903096,PMI: Skip,903096,PMI: Skip,1177221,I prefer not to answer


In [9]:
DATA_BUCKET = '/home/jupyter/workspace/data_bucket'
GENETIC_FOLDER = f'{DATA_BUCKET}/v9_gen_data'
dataset_08947253_person_df.to_csv(f"{GENETIC_FOLDER}/df_whole_DS_v9.tsv", sep = "\t", index=False)
dataset_hl = (hl.import_table(f"{GENETIC_FOLDER}/df_whole_DS_v9.tsv",
                              types={'person_id':hl.tstr},
                              impute=True,
                              key='person_id')
             )

KeyboardInterrupt: 

# Genetic Data

In [4]:
vat_path = "/home/jupyter/workspace/cdrv9/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/aux/vat/vat_complete.bgz.tsv.gz"
vat_path
vat_table = hl.import_table(
    vat_path, 
    force=True, 
    quote='"', 
    delimiter="\t", 
    force_bgz=True, 
    types={"position": hl.tint, "contig": hl.tstr, "ref_allele": hl.tstr, "alt_allele": hl.tstr}
)
vat_table.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'vid': str 
    'transcript': str 
    'contig': str 
    'position': int32 
    'ref_allele': str 
    'alt_allele': str 
    'gvs_all_ac': str 
    'gvs_all_an': str 
    'gvs_all_af': str 
    'gvs_all_sc': str 
    'gvs_max_af': str 
    'gvs_max_ac': str 
    'gvs_max_an': str 
    'gvs_max_sc': str 
    'gvs_max_subpop': str 
    'gvs_afr_ac': str 
    'gvs_afr_an': str 
    'gvs_afr_af': str 
    'gvs_afr_sc': str 
    'gvs_amr_ac': str 
    'gvs_amr_an': str 
    'gvs_amr_af': str 
    'gvs_amr_sc': str 
    'gvs_eas_ac': str 
    'gvs_eas_an': str 
    'gvs_eas_af': str 
    'gvs_eas_sc': str 
    'gvs_eur_ac': str 
    'gvs_eur_an': str 
    'gvs_eur_af': str 
    'gvs_eur_sc': str 
    'gvs_mid_ac': str 
    'gvs_mid_an': str 
    'gvs_mid_af': str 
    'gvs_mid_sc': str 
    'gvs_oth_ac': str 
    'gvs_oth_an': str 
    'gvs_oth_af': str 
    'gvs_oth_sc

In [5]:
transcripts_of_interest = [
    "ENST00000279146", "ENST00000389048", "ENST00000257430", "ENST00000675843", "ENST00000307078",
    "ENST00000460680", "ENST00000260947", "ENST00000355112", "ENST00000372037", "ENST00000357654",
    "ENST00000380152", "ENST00000259008", "ENST00000639785", "ENST00000367435", "ENST00000261769",
    "ENST00000257904", "ENST00000228872", "ENST00000313407", "ENST00000440480", "ENST00000304494",
    "ENST00000579755", "ENST00000498907", "ENST00000404276", "ENST00000302763", "ENST00000343455",
    "ENST00000325385", "ENST00000275493", "ENST00000263735", "ENST00000366560", "ENST00000285071",
    "ENST00000341105", "ENST00000487848", "ENST00000370818", "ENST00000651081", "ENST00000300177",
    "ENST00000651154", "ENST00000290295", "ENST00000610977", "ENST00000311189", "ENST00000417302",
    "ENST00000288135", "ENST00000358664", "ENST00000555147", "ENST00000450708", "ENST00000397752",
    "ENST00000394351", "ENST00000352241", "ENST00000231790", "ENST00000233146", "ENST00000265081",
    "ENST00000234420", "ENST00000456914", "ENST00000710952", "ENST00000265433", "ENST00000358273",
    "ENST00000338641", "ENST00000651570", "ENST00000261584", "ENST00000257290", "ENST00000226382",
    "ENST00000265849", "ENST00000440232", "ENST00000320574", "ENST00000357628", "ENST00000589228",
    "ENST00000331920", "ENST00000437951", "ENST00000644628", "ENST00000371953", "ENST00000378823",
    "ENST00000337432", "ENST00000345365", "ENST00000267163", "ENST00000617875", "ENST00000355710",
    "ENST00000675419", "ENST00000264932", "ENST00000301761", "ENST00000375499", "ENST00000367975",
    "ENST00000375549", "ENST00000342988", "ENST00000344626", "ENST00000646693", "ENST00000618915",
    "ENST00000644036", "ENST00000348513", "ENST00000326873", "ENST00000369902", "ENST00000310581",
    "ENST00000258439", "ENST00000269305", "ENST00000298552", "ENST00000219476", "ENST00000256474",
    "ENST00000298139", "ENST00000452863"
]
transcript_vat_table = vat_table.filter(
    hl.any(lambda t: vat_table.transcript.startswith(t), hl.literal(transcripts_of_interest))
)

In [6]:
lst = [
    'pathogenic', 
    'likely pathogenic', 
    'likely pathogenic, pathogenic',  
    'likely pathogenic, affects',
    'likely pathogenic, association',
    'likely pathogenic, drug response',
    'likely pathogenic, drug response, not provided',
    'likely pathogenic, not provided',
    'likely pathogenic, other',
    'likely pathogenic, pathogenic, affects',
    'likely pathogenic, pathogenic, association',
    'likely pathogenic, pathogenic, drug response',
    'likely pathogenic, pathogenic, drug response, not provided',
    'likely pathogenic, pathogenic, drug response, other',
    'likely pathogenic, pathogenic, drug response, protective',
    'likely pathogenic, pathogenic, drug response, risk factor',
    'likely pathogenic, pathogenic, not provided',
    'likely pathogenic, pathogenic, other',
    'likely pathogenic, pathogenic, protective',
    'likely pathogenic, pathogenic, risk factor',
    'likely pathogenic, pathogenic, risk factor, not provided',
    'likely pathogenic, risk factor',
    'likely risk allele',
    'pathogenic, association',
    'pathogenic, association, protective',
    'pathogenic, confers sensitivity',
    'pathogenic, drug response',
    'pathogenic, drug response, not provided',
    'pathogenic, drug response, risk factor',
    'pathogenic, drug response, risk factor, protective',
    'pathogenic, not provided',
    'pathogenic, other',
    'pathogenic, protective',
    'pathogenic, protective, other',
    'pathogenic, risk factor',
    'pathogenic, risk factor, not provided',
    'pathogenic, risk factor, other',
    'pathogenic, risk factor, protective',
    'risk factor'
]

# Filter vat_table to only contain Clinvar classification in lst
filtered_vat_table = transcript_vat_table.filter(hl.set(lst).contains(transcript_vat_table.clinvar_classification))

# Create locus and alleles to match MatrixTable key structure
filtered_vat_table = filtered_vat_table.annotate(
    locus=hl.locus(filtered_vat_table.contig, filtered_vat_table.position, reference_genome='GRCh38'),
    alleles=[filtered_vat_table.ref_allele, filtered_vat_table.alt_allele]
)
filtered_vat_table = filtered_vat_table.key_by('locus', 'alleles')

# List of consequences to exclude
exclude_consequences = ['downstream_gene_variant', 'upstream_gene_variant']

# Filter the table to exclude rows with these consequences
filtered_vat_table = filtered_vat_table.filter(
    ~hl.set(exclude_consequences).contains(filtered_vat_table.consequence)
)


In [9]:
import hail as hl

mt_wgs_clinvar_path = (
    "/home/jupyter/workspace/cdrv9/"
    "vwb-aou-datasets-controlled-v9/v9/wgs/short_read/"
    "snpindel/clinvar/splitMT/hail.mt"
)

genes_of_interest = {
    "AIP", "ALK", "APC", "ATM", "AXIN2", "BAP1", "BARD1", "BLM",
    "BMPR1A", "BRCA1", "BRCA2", "BRIP1", "CASR", "CDC73", "CDH1",
    "CDK4", "CDKN1B", "CDKN1C", "CDKN2A", "CEBPA", "CHEK2",
    "CTNNA1", "DICER1", "DIS3L2", "EGFR", "EPCAM", "FH", "FLCN",
    "GATA2", "GPC3", "GREM1", "HOXB13", "HRAS", "KIT", "MAX",
    "MC1R", "MEN1", "MET", "MITF", "MLH1", "MSH2", "MSH3", "MSH6",
    "MUTYH", "NBN", "NF1", "NF2", "NTHL1", "PALB2", "PDGFRA",
    "PHOX2B", "PMS2", "POLD1", "POLE", "POT1", "PRKAR1A", "PTCH1",
    "PTEN", "RAD50", "RAD51C", "RAD51D", "RB1", "RECQL4", "RET",
    "RUNX1", "SDHA", "SDHAF2", "SDHB", "SDHC", "SDHD", "SMAD4",
    "SMARCA4", "SMARCB1", "SMARCE1", "STK11", "SUFU", "TERC", "TERT",
    "TMEM127", "TP53", "TSC1", "TSC2", "VHL", "WRN", "WT1"
}

# Read MatrixTable
mt = hl.read_matrix_table(mt_wgs_clinvar_path)

# Keep only cohort samples and selected variants
mt = mt.semi_join_cols(dataset_hl)
mt = mt.semi_join_rows(filtered_vat_table)

# Add sample metadata
mt = mt.annotate_cols(
    metadata=dataset_hl[mt.s]
)

# Add only the row annotation actually needed
variant_annotations = filtered_vat_table.select("gene_symbol")

mt = mt.annotate_rows(
    gene_symbol=variant_annotations[mt.locus, mt.alleles].gene_symbol
)

# Keep selected genes
genes_literal = hl.literal(genes_of_interest)

mt = mt.filter_rows(
    hl.is_defined(mt.gene_symbol) &
    genes_literal.contains(mt.gene_symbol)
)

# Filter non-reference entries BEFORE calling entries()
mt = mt.filter_entries(
    hl.is_defined(mt.GT) &
    mt.GT.is_non_ref()
)

# Write a filtered MatrixTable checkpoint
filtered_mt_path = f"{GENETIC_FOLDER}/filtered_nonref.mt"
mt = mt.checkpoint(
    filtered_mt_path,
    overwrite=True
)

In [10]:
# Load the existing filtered MatrixTable
mt = hl.read_matrix_table(f"{GENETIC_FOLDER}/filtered_nonref.mt")

# Restore all VAT annotation fields
mt = mt.annotate_rows(
    annotations=filtered_vat_table[mt.locus, mt.alleles]
)

# Restore the V8 column, if needed
mt = mt.annotate_entries(
    has_variant=hl.is_defined(mt.GT) & mt.GT.is_non_ref()
)

# Recreate only the entries table
entries_table = mt.entries()

# Convert and save
entries_table_df = entries_table.to_pandas()

entries_table_df.to_csv(
    f"{GENETIC_FOLDER}/entries_table_full_v9.csv",
    index=False
)

print(entries_table_df.shape)

2026-07-12 20:09:48.703 package$: WARN: entries(): Resulting entries table is sorted by '(row_key, col_key)'.
    To preserve row-major matrix table order, first unkey columns with 'key_cols_by()'
2026-07-13 00:24:08.086 DAGScheduler: WARN: Broadcasting large task binary with size 127.4 MiB
2026-07-13 00:24:08.537 TaskSetManager: WARN: Stage 5 contains a task of very large size (130417 KiB). The maximum recommended task size is 1000 KiB.
IOPub message rate exceeded.                              (472 + 5142) / 119489]
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.=======>                   (68524 + 24748) / 119489]
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To cha

In [ ]:
mt = hl.read_matrix_table(filtered_mt_path)
# Extract only needed fields
entries_table = mt.entries()

# Write distributed Hail Table first
entries_ht_path = f"{GENETIC_FOLDER}/entries_table_v9.ht"

entries_table = entries_table.checkpoint(
    entries_ht_path,
    overwrite=True
)

# Export as multiple compressed TSV shards
entries_table.export(
    f"{GENETIC_FOLDER}/entries_table_v9.tsv.bgz",
    parallel="header_per_shard"
)

2026-07-11 14:36:59.083 package$: WARN: entries(): Resulting entries table is sorted by '(row_key, col_key)'.
    To preserve row-major matrix table order, first unkey columns with 'key_cols_by()'
2026-07-11 14:37:10.852 DAGScheduler: WARN: Broadcasting large task binary with size 100.9 MiB
2026-07-11 14:37:11.314 TaskSetManager: WARN: Stage 14 contains a task of very large size (103281 KiB). The maximum recommended task size is 1000 KiB.
IOPub message rate exceeded.====>                        (57830 + 119) / 119489]
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.=====================>        (98098 + 96) / 119489]
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To ch

In [5]:
entries_ht_path = f"{GENETIC_FOLDER}/entries_table_v9.ht"
entries_table = hl.read_table(entries_ht_path)
entries_table_df = entries_table.to_pandas()

entries_table_df.to_csv(f'{GENETIC_FOLDER}/entries_table_full_v9.csv', index=False)

2026-07-12 06:13:31.111 DAGScheduler: WARN: Broadcasting large task binary with size 20.0 MiB
2026-07-12 06:13:31.576 TaskSetManager: WARN: Stage 0 contains a task of very large size (20486 KiB). The maximum recommended task size is 1000 KiB.
IOPub message rate exceeded.                           (16263 + 82293) / 119489]
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [7]:
entries = pd.read_csv(f"{GENETIC_FOLDER}/entries_table_full_v9.csv")

In [15]:
entries[entries['gene_symbol']=='SDHB']

,locus,alleles,filters,a_index,was_split,variant_qc.gq_stats.mean,variant_qc.gq_stats.stdev,variant_qc.gq_stats.min,variant_qc.gq_stats.max,variant_qc.call_rate,...,metadata.ethnicity_concept_id,metadata.ethnicity,metadata.sex_at_birth_concept_id,metadata.sex_at_birth,GQ,FT,PS,RGQ,AD,GT
0,chr1:17018936,"['A', 'ATAGC']",set(),1,False,39.988253,0.390293,20.0,48.0,0.999918,...,38003563,Hispanic or Latino,45878463,Female,48,PASS,NaN,79,"[17, 16]",0/1
1,chr1:17022615,"['C', 'T']",set(),2,True,39.993679,0.286187,20.0,48.0,0.999959,...,38003564,Not Hispanic or Latino,45878463,Female,48,PASS,NaN,85,"[19, 20]",0/1
2,chr1:17022615,"['C', 'T']",set(),2,True,39.993679,0.286187,20.0,48.0,0.999959,...,38003564,Not Hispanic or Latino,45878463,Female,48,PASS,NaN,84,"[19, 16]",0/1
3,chr1:17022627,"['C', 'A']",set(),1,False,39.991987,0.319945,20.0,48.0,0.999938,...,38003564,Not Hispanic or Latino,45878463,Female,48,PASS,NaN,84,"[23, 19]",0/1
4,chr1:17022649,"['G', 'T']",set(),1,False,39.989855,0.357631,20.0,48.0,0.999937,...,38003564,Not Hispanic or Latino,45878463,Female,47,PASS,NaN,85,"[16, 23]",0/1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,chr1:17044848,"['C', 'CGG']",set(),1,True,39.990918,0.373866,20.0,48.0,0.999832,...,38003563,Hispanic or Latino,45878463,Female,48,PASS,NaN,79,"[22, 23]",0/1
73,chr1:17044890,"['T', 'A']",set(),1,True,39.989381,0.372728,20.0,46.0,0.999923,...,38003564,Not Hispanic or Latino,45880669,Male,46,PASS,NaN,82,"[17, 11]",0/1
74,chr1:17053998,"['AG', 'A']",set(),1,True,39.994524,0.272746,20.0,46.0,0.999922,...,38003564,Not Hispanic or Latino,45880669,Male,46,PASS,17053998.0,79,"[28, 23]",0|1
75,chr1:17054002,"['G', 'GGCAACCGGCGCCTCAAGGAGAGT']",set(),2,True,93.626240,14.435231,7.0,99.0,1.000000,...,38003564,Not Hispanic or Latino,45878463,Female,50,PASS,NaN,655,"[15, 19]",0/1


In [9]:
mt_wgs_clinvar_path = '/home/jupyter/workspace/cdrv9/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/clinvar/splitMT/hail.mt'

mt = hl.read_matrix_table(mt_wgs_clinvar_path)
mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'filters': set<str>
    'a_index': int32
    'was_split': bool
    'variant_qc': struct {
        gq_stats: struct {
            mean: float64, 
            stdev: float64, 
            min: float64, 
            max: float64
        }, 
        call_rate: float64, 
        n_called: int64, 
        n_not_called: int64, 
        n_filtered: int64, 
        n_het: int64, 
        n_non_ref: int64, 
        het_freq_hwe: float64, 
        p_value_hwe: float64, 
        p_value_excess_het: float64
    }
    'info': struct {
        AC: array<int32>, 
        AF: array<float64>, 
        AN: int32, 
        homozygote_count: array<int32>
    }
----------------------------------------
Entry fields:
    'GQ': int32
    'FT': str
    'PS': int64
    'RG

In [10]:
# Select only the samples in dataset_hl
mt_sub = mt.semi_join_cols(dataset_hl)
# Select only the variants in filtered_vat_table
mt_sub = mt_sub.semi_join_rows(filtered_vat_table)
mt_sub.describe()

# Annotate MatrixTable columns with metadata
mt_sub = mt_sub.annotate_cols(metadata=dataset_hl[mt_sub.s])
# Annotate MatrixTable rows with variant annotations
mt_sub = mt_sub.annotate_rows(annotations=filtered_vat_table[mt_sub.locus, mt_sub.alleles])
# Extract the column fields (metadata and sample ID 's') into a Hail Table
metadata_table = mt_sub.cols()

# Convert the Hail Table to a Pandas DataFrame
metadata_df = metadata_table.to_pandas()

# Save the DataFrame to a CSV file
metadata_df.to_csv(f'{GENETIC_FOLDER}/metadata_v9.csv', index=False)

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'filters': set<str>
    'a_index': int32
    'was_split': bool
    'variant_qc': struct {
        gq_stats: struct {
            mean: float64, 
            stdev: float64, 
            min: float64, 
            max: float64
        }, 
        call_rate: float64, 
        n_called: int64, 
        n_not_called: int64, 
        n_filtered: int64, 
        n_het: int64, 
        n_non_ref: int64, 
        het_freq_hwe: float64, 
        p_value_hwe: float64, 
        p_value_excess_het: float64
    }
    'info': struct {
        AC: array<int32>, 
        AF: array<float64>, 
        AN: int32, 
        homozygote_count: array<int32>
    }
----------------------------------------
Entry fields:
    'GQ': int32
    'FT': str
    'PS': int64
    'RG

2026-07-10 01:32:52.900 package$: WARN: cols(): Resulting column table is sorted by 'col_key'.
    To preserve matrix table column order, first unkey columns with 'key_cols_by()'
                                                                                

In [11]:
# Kind of redundant though
# Filter the MatrixTable to specific genes
genes_of_interest = ['AIP', 'ALK', 'APC', 'ATM', 'AXIN2', 'BAP1', 'BARD1', 'BLM', 'BMPR1A', 'BRCA1', 'BRCA2', 
                     'BRIP1', 'CASR', 'CDC73', 'CDH1', 'CDK4', 'CDKN1B', 'CDKN1C', 'CDKN2A', 'CEBPA', 'CHEK2', 
                     'CTNNA1', 'DICER1', 'DIS3L2', 'EGFR', 'EPCAM', 'FH', 'FLCN', 'GATA2', 'GPC3', 'GREM1', 
                     'HOXB13', 'HRAS', 'KIT', 'MAX', 'MC1R', 'MEN1', 'MET', 'MITF', 'MLH1', 'MSH2', 'MSH3', 
                     'MSH6', 'MUTYH', 'NBN', 'NF1', 'NF2', 'NTHL1', 'PALB2', 'PDGFRA', 'PHOX2B', 'PMS2', 
                     'POLD1', 'POLE', 'POT1', 'PRKAR1A', 'PTCH1', 'PTEN', 'RAD50', 'RAD51C', 'RAD51D', 'RB1', 
                     'RECQL4', 'RET', 'RUNX1', 'SDHA', 'SDHAF2', 'SDHB', 'SDHC', 'SDHD', 'SMAD4', 'SMARCA4', 
                     'SMARCB1', 'SMARCE1', 'STK11', 'SUFU', 'TERC', 'TERT', 'TMEM127', 'TP53', 'TSC1', 'TSC2', 
                     'VHL', 'WRN', 'WT1']

mt_filtered = mt_sub.filter_rows(hl.literal(genes_of_interest).contains(mt_sub.annotations.gene_symbol))
mt_filtered = mt_filtered.annotate_entries(has_variant=mt_filtered.GT.is_non_ref())
mt_filtered.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'metadata': struct {
        gender_concept_id: int32, 
        gender: str, 
        date_of_birth: str, 
        race_concept_id: int32, 
        race: str, 
        ethnicity_concept_id: int32, 
        ethnicity: str, 
        sex_at_birth_concept_id: int32, 
        sex_at_birth: str
    }
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'filters': set<str>
    'a_index': int32
    'was_split': bool
    'variant_qc': struct {
        gq_stats: struct {
            mean: float64, 
            stdev: float64, 
            min: float64, 
            max: float64
        }, 
        call_rate: float64, 
        n_called: int64, 
        n_not_called: int64, 
        n_filtered: int64, 
        n_het: int64, 
        n_non_ref: int64, 
        het_freq_hwe: float64, 
        p_value_hwe: fl

In [12]:
entries_table = mt_filtered.entries()
# Filter entries to include only non-reference variants
entries_table = entries_table.filter(entries_table.has_variant)
entries_table.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'filters': set<str> 
    'a_index': int32 
    'was_split': bool 
    'variant_qc': struct {
        gq_stats: struct {
            mean: float64, 
            stdev: float64, 
            min: float64, 
            max: float64
        }, 
        call_rate: float64, 
        n_called: int64, 
        n_not_called: int64, 
        n_filtered: int64, 
        n_het: int64, 
        n_non_ref: int64, 
        het_freq_hwe: float64, 
        p_value_hwe: float64, 
        p_value_excess_het: float64
    } 
    'info': struct {
        AC: array<int32>, 
        AF: array<float64>, 
        AN: int32, 
        homozygote_count: array<int32>
    } 
    'annotations': struct {
        vid: str, 
        transcript: str, 
        contig: str, 
        position: int32, 
        ref_allele: str, 
        alt_allele: str

2026-07-10 01:33:29.779 package$: WARN: entries(): Resulting entries table is sorted by '(row_key, col_key)'.
    To preserve row-major matrix table order, first unkey columns with 'key_cols_by()'


# Analysis

In [12]:
entries_v9 = pd.read_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_full_v9.csv')
entries_v8 = pd.read_csv('/home/jupyter/workspace/genetic_bucket/data/entries_table_full_v8.csv')

/tmp/ipykernel_1035/1122821914.py:1: DtypeWarning:

Columns (68,72,82,127,132,133,134,135,136) have mixed types. Specify dtype option on import or set low_memory=False.

/tmp/ipykernel_1035/1122821914.py:2: DtypeWarning:

Columns (67,71,81) have mixed types. Specify dtype option on import or set low_memory=False.



In [16]:
entries_v9.columns.values

array(['locus', 'alleles', 'filters', 'a_index', 'was_split',
       'variant_qc.gq_stats.mean', 'variant_qc.gq_stats.stdev',
       'variant_qc.gq_stats.min', 'variant_qc.gq_stats.max',
       'variant_qc.call_rate', 'variant_qc.n_called',
       'variant_qc.n_not_called', 'variant_qc.n_filtered',
       'variant_qc.n_het', 'variant_qc.n_non_ref',
       'variant_qc.het_freq_hwe', 'variant_qc.p_value_hwe',
       'variant_qc.p_value_excess_het', 'info.AC', 'info.AF', 'info.AN',
       'info.homozygote_count', 'gene_symbol', 'annotations.vid',
       'annotations.transcript', 'annotations.contig',
       'annotations.position', 'annotations.ref_allele',
       'annotations.alt_allele', 'annotations.gvs_all_ac',
       'annotations.gvs_all_an', 'annotations.gvs_all_af',
       'annotations.gvs_all_sc', 'annotations.gvs_max_af',
       'annotations.gvs_max_ac', 'annotations.gvs_max_an',
       'annotations.gvs_max_sc', 'annotations.gvs_max_subpop',
       'annotations.gvs_afr_ac', 'annot